In [1]:
import os
import pandas as pd
import pygrib
import tempfile
import bz2

# Directory containing downloaded GRIB files
input_directory = "downloaded_grib_files"

# Output CSV filename
output_csv = "parsed_data.csv"

# List to store parsed data
data = []

# Loop through the GRIB files in the input directory
for filename in os.listdir(input_directory):
    if filename.endswith(".grib2.bz2"):
        file_path = os.path.join(input_directory, filename)

        # Open the compressed GRIB2 file using bz2
        with bz2.open(file_path, "rb") as f:
            # Decompress the content and read it using pygrib
            decompressed_data = f.read()

            # Create a temporary file to store the decompressed data
            with tempfile.NamedTemporaryFile(delete=False) as temp_file:
                temp_file.write(decompressed_data)
                temp_file_path = temp_file.name

            # Open the decompressed data using pygrib
            grbs = pygrib.open(temp_file_path)

            # Iterate through the messages in the GRIB file
            for grb in grbs:
                # Extract relevant information
                parameter_name = grb.parameterName
                level_type = grb.levelType
                level = grb.level
                validity_date = grb.validityDate
                validity_time = grb.validityTime
                values_min = grb.values.min()
                values_max = grb.values.max()

                # Append the information to the data list
                data.append([parameter_name, level_type, level, validity_date, validity_time, values_min, values_max])

            # Close the GRIB file
            grbs.close()

            # Clean up: Delete the temporary file
            os.remove(temp_file_path)

# Create a DataFrame from the parsed data
columns = ["Parameter Name", "Level Type", "Level", "Valid Date", "Valid Time", "Values Min", "Values Max"]
df = pd.DataFrame(data, columns=columns)

# Save the DataFrame to a CSV file
df.to_csv(output_csv, index=False)

print(f"Data saved to '{output_csv}'")


Data saved to 'parsed_data.csv'
